# espnet_onnx ASR demonstration

This notebook provides a demonstration of how to export your trained ASR model into onnx format.

see also:
- ESPnet: https://github.com/espnet/espnet
- espnet_onnx: https://github.com/Masao-Someki/espnet_onnx

Author: [Masao Someki](https://github.com/Masao-Someki)


## Table of Contents

- Install Dependency
- Export your model
- Inference with onnx
- Using streaming model

# Install Dependency

Use a fresh Python 3.12 or 3.13 kernel (the repository CI uses Python 3.13).
In Colab, select a compatible runtime before installing dependencies.

These demos use the current repository implementation and its export requirements:
- torch >= 2.11.0 and a compatible torchaudio version
- espnet >= 202611 and espnet_model_zoo
- numpy >= 2.0.0, librosa >= 0.10.2, onnx >= 1.22.0, and onnxscript

Run the installation cell once, then restart the kernel/session before continuing.
The espnet_onnx source revision is pinned so the demo uses the exporter shown here.


In [ ]:
%pip install --upgrade "espnet_onnx @ git+https://github.com/espnet/espnet_onnx.git@1015bea4f4a2f8a0b70629e8f0ad06162b143bee" "espnet>=202611" espnet_model_zoo "numpy>=2.0.0" "librosa>=0.10.2" "onnx>=1.22.0" onnxscript "protobuf>=5.29.6" scipy pyworld
%pip check

# Export your model


## Export model from espnet_model_zoo

The easiest way to export a model is to use `espnet_model_zoo`. You can download, unpack, and export the pretrained models with `export_from_pretrained` method.
`espnet_onnx` will save the onnx models into cache directory, which is `${HOME}/.cache/espnet_onnx` in default.

In [ ]:
# export the model.
from espnet_onnx.export import ASRModelExport

tag_name = 'kamo-naoyuki/timit_asr_train_asr_raw_word_valid.acc.ave'

m = ASRModelExport()
m.export_from_pretrained(tag_name)

## Export from custom model

`espnet_onnx` can also export your own trained model with `export` method.

The following script shows how to export from `espnet2.bin.asr_inference.Speech2Text` instance.
You can also export from a zipped file, by using the `export_from_zip` function.  
For this demonstration, I'm using the `from_pretrained` method to load parameters, but you can load your own model.

In [ ]:
# prepare the espnet2.bin.asr_inference.Speech2Text instance.
from espnet2.bin.asr_inference import Speech2Text

tag_name = 'kamo-naoyuki/timit_asr_train_asr_raw_word_valid.acc.ave'
speech2text = Speech2Text.from_pretrained(tag_name)


# export model
from espnet_onnx.export import ASRModelExport

sample_model_tag = 'demo/sample_model_1'
m = ASRModelExport()
m.export(
    speech2text,
    sample_model_tag,
    quantize=False
)

# Inference with onnx

Now, let's use the exported models for inference.

In [ ]:
# please provide the tag_name to specify exported model.
tag_name = 'kamo-naoyuki/timit_asr_train_asr_raw_word_valid.acc.ave'


# upload wav file and let's inference!
import librosa
from google.colab import files

wav_file = files.upload()
y, sr = librosa.load(list(wav_file.keys())[0], sr=16000)


# Use the exported onnx file to inference.
from espnet_onnx import Speech2Text

speech2text = Speech2Text(tag_name)
nbest = speech2text(y)
print(nbest[0][0])

# Using streaming model

Model exportation is exactly the same as non-streaming model. You can follow the `#Export your model` chapter.

As for streaming, you can specify the following configuration additionaly. Usually, these values should be the same as the training configuration.
- block_size
- hop_size
- look_ahead

Pass consecutive mono audio chunks without padding each chunk. The streaming API
buffers the first encoder window and retains STFT overlap internally. Intermediate
calls may return an empty list while waiting for enough audio. Call `end()` after
the final chunk to flush the remaining samples and obtain the final result.

Now, let's demonstrate the streaming inference.

In [ ]:
# Export the streaming model.
# Note that the following model is very large
from espnet_onnx.export import ASRModelExport

tag_name = 'D-Keqi/espnet_asr_train_asr_streaming_transformer_raw_en_bpe500_sp_valid.acc.ave'

m = ASRModelExport()
m.export_from_pretrained(tag_name)

In [ ]:
# In this tutorial, we will use the recorded wav file to simulate streaming.
import librosa
from espnet_onnx import StreamingSpeech2Text

tag_name = 'D-Keqi/espnet_asr_train_asr_streaming_transformer_raw_en_bpe500_sp_valid.acc.ave'
streaming_model = StreamingSpeech2Text(tag_name)

# upload wav file
from google.colab import files
wav_file = files.upload()
y, sr = librosa.load(list(wav_file.keys())[0], sr=16000)

# Feed consecutive audio chunks, as a microphone would provide them.
# The first encoder window is buffered internally; early results may be empty.
streaming_model.start()
for start in range(0, len(y), streaming_model.hop_size):
    nbest = streaming_model(y[start : start + streaming_model.hop_size])
    if nbest:
        print(f'Partial result : {nbest[0][0]}')

# Process the remaining audio and look-ahead, then finalize recognition.
final_nbest = streaming_model.end()
if final_nbest:
    print(f'Final result : {final_nbest[0][0]}')
else:
    print('No hypothesis was returned for this recording.')